# TUẦN 9 — ĐÁNH GIÁ THỰC NGHIỆM PHISHGUARDVN

**Mục tiêu:** rà soát pipeline, đánh giá mô hình trên Test độc lập, kiểm thử robustness và ghi nhận thời gian suy luận.

> Nguyên tắc: chỉ chọn mô hình, siêu tham số, alpha và threshold trên Validation. Test chỉ được dùng một lần cho đánh giá cuối cùng. Không thay nhãn proxy thành nhãn phishing. Không tự tạo checkpoint hoặc số liệu còn thiếu.

Notebook này được đặt trong `PhishGuardVN_Weeks5_8/notebooks/`. Khi chạy, thư mục làm việc có thể là thư mục dự án hoặc `notebooks/`.


## 0. Phạm vi và giới hạn dữ liệu hiện có

Qua kiểm tra gói dự án được cung cấp:

- Có mô hình URL XGBoost và manifest 31 đặc trưng lexical.
- Có baseline văn bản TF-IDF + Logistic Regression nhị phân trên nhãn Spam/Scam nguồn.
- Chưa có checkpoint XLM-RoBERTa và chưa có artifact phân loại văn bản ba lớp.
- Risk Fusion hiện tại được ghi rõ là `proxy_only_not_validated_phishing_probability`, với alpha Validation bằng 1.0.
- Có URL split manifest gồm Train/Validation/Test; notebook sẽ đánh giá Test của nhánh URL.
- Không có bằng chứng nhãn phishing đã được người gán nhãn rà soát cho các cặp SMS–URL. Vì vậy không báo cáo Text+URL như một benchmark phishing đã được xác nhận.

Các mục chưa đủ đầu vào sẽ được đánh dấu **BLOCKED / NOT EVALUATED**, không điền số liệu giả.


In [ ]:
from pathlib import Path
import json, time, re, unicodedata, sys, platform
import numpy as np
import pandas as pd

# Tìm thư mục gốc dự án, hỗ trợ chạy từ root hoặc notebooks/
CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "phishguard").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục phishguard/. Hãy đặt notebook trong notebooks/ của dự án.")
sys.path.insert(0, str(PROJECT_ROOT))

from phishguard.features import feature_frame, LEXICAL, extract_urls
from phishguard.training import text_input
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, average_precision_score)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
import joblib

MODELS = PROJECT_ROOT / "models"
DATA = PROJECT_ROOT / "data"
REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
manifest = json.loads((MODELS / "manifest.json").read_text(encoding="utf-8"))
print("Project:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("Manifest version:", manifest.get("versions"))


## 1. Rà soát nhãn và thứ tự đặc trưng URL

Theo manifest hiện tại, nhãn gốc PhiUSIIL là `0 = Phishing`, `1 = Legitimate`; mã huấn luyện đảo thành `y_phishing = 1 - label`, tức nội bộ `1 = Phishing`.

Kiểm tra dưới đây xác nhận thứ tự 31 đặc trưng trong manifest phải trùng khớp với thứ tự `LEXICAL` và DataFrame khi inference.


In [ ]:
expected_mapping = {"PhiUSIIL_0": "Phishing", "PhiUSIIL_1": "Legitimate"}
assert manifest.get("source_label_mapping") == expected_mapping, manifest.get("source_label_mapping")
assert manifest.get("feature_columns") == list(LEXICAL), "Thứ tự feature manifest khác LEXICAL!"
assert len(LEXICAL) == 31, f"Số feature không đúng 31: {len(LEXICAL)}"

probe = feature_frame(["https://example.com/login?next=account"])
assert list(probe.columns) == list(LEXICAL)
assert probe.shape == (1, 31)
assert np.isfinite(probe.to_numpy()).all()

print("PASS: mapping nhãn nguồn và mapping nội bộ đã khớp.")
print("PASS: 31 feature, đúng thứ tự manifest == LEXICAL == feature_frame.")
print("Feature order:", list(LEXICAL))


## 2. Kiểm tra split URL và loại trừ rò rỉ theo domain

Sử dụng manifest đã được tạo trước đó. Kiểm tra không có `domain_group` giao nhau giữa Train, Validation và Test. Không thay đổi split trong giai đoạn đánh giá.


In [ ]:
split_path = DATA / "url_split_manifest.csv.gz"
if not split_path.exists():
    raise FileNotFoundError(f"Thiếu {split_path}")
url_df = pd.read_csv(split_path, compression="gzip")
required = {"canonical", "domain_group", "label", "y_phishing", "split"}
assert required.issubset(url_df.columns)
assert set(url_df["split"].unique()) == {"train", "validation", "test"}
assert set(url_df["y_phishing"].dropna().unique()) == {0, 1}
assert (url_df["y_phishing"].astype(int) == 1 - url_df["label"].astype(int)).all()

groups = {s: set(url_df.loc[url_df.split.eq(s), "domain_group"].dropna()) for s in ["train","validation","test"]}
assert not (groups["train"] & groups["validation"])
assert not (groups["train"] & groups["test"])
assert not (groups["validation"] & groups["test"])

split_summary = url_df.groupby(["split","y_phishing"]).size().unstack(fill_value=0)
display(split_summary)
print("PASS: không có domain_group giao nhau giữa ba tập.")


## 3. Đánh giá XGBoost URL trên Internal Test

XGBoost đã được chọn trên Validation ở giai đoạn trước. Threshold được khóa theo `manifest.json`; không tối ưu lại trên Test. Confusion Matrix theo thứ tự `[[TN, FP], [FN, TP]]`.


In [ ]:
from xgboost import XGBClassifier

url_model = XGBClassifier()
url_model.load_model(str(MODELS / "url_model.json"))
test_df = url_df[url_df.split.eq("test")].copy()
X_test = feature_frame(test_df["canonical"].tolist())
y_test = test_df["y_phishing"].astype(int).to_numpy()

assert list(X_test.columns) == manifest["feature_columns"]
assert X_test.shape[1] == int(url_model.n_features_in_) if hasattr(url_model, "n_features_in_") else X_test.shape[1] == len(LEXICAL)

t0 = time.perf_counter()
p_test = url_model.predict_proba(X_test)[:, 1]
elapsed = time.perf_counter() - t0
threshold_url = float(manifest["url_threshold"])
pred_test = (p_test >= threshold_url).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred_test, labels=[0,1]).ravel()

url_test_metrics = {
    "model": manifest.get("url_selected", "XGBoost"),
    "threshold_locked_from_validation": threshold_url,
    "n_test": int(len(y_test)),
    "precision": float(precision_score(y_test, pred_test, zero_division=0)),
    "recall": float(recall_score(y_test, pred_test, zero_division=0)),
    "f1": float(f1_score(y_test, pred_test, zero_division=0)),
    "fpr": float(fp / max(1, fp + tn)),
    "pr_auc_average_precision": float(average_precision_score(y_test, p_test)),
    "confusion_matrix_TN_FP_FN_TP": [[int(tn),int(fp)],[int(fn),int(tp)]],
    "total_inference_seconds": float(elapsed),
    "mean_inference_ms_per_url": float(elapsed * 1000 / max(1,len(y_test))),
}
url_test_metrics


## 4. Baseline Logistic Regression URL — chọn threshold trên Validation, đánh giá Test một lần

So sánh LR và XGBoost trên cùng 31 đặc trưng, cùng split. Hai cấu hình C được xem xét trên Validation; cấu hình được chọn theo F1, sau đó Recall, sau đó FPR. Threshold cũng được chọn trên Validation. Test không tham gia chọn C hoặc threshold.


In [ ]:
train_df = url_df[url_df.split.eq("train")].copy()
val_df = url_df[url_df.split.eq("validation")].copy()

X_train = feature_frame(train_df["canonical"].tolist())
y_train = train_df["y_phishing"].astype(int).to_numpy()
X_val = feature_frame(val_df["canonical"].tolist())
y_val = val_df["y_phishing"].astype(int).to_numpy()

def calc_metrics(y, p, threshold):
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "precision": float(precision_score(y,pred,zero_division=0)),
        "recall": float(recall_score(y,pred,zero_division=0)),
        "f1": float(f1_score(y,pred,zero_division=0)),
        "fpr": float(fp/max(1,fp+tn)),
        "pr_auc_average_precision": float(average_precision_score(y,p)),
        "threshold": float(threshold),
        "confusion_matrix_TN_FP_FN_TP": [[int(tn),int(fp)],[int(fn),int(tp)]],
    }

lr_candidates = []
for C in [0.1, 1.0]:
    lr = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1500, random_state=42))
    lr.fit(X_train, y_train)
    pv = lr.predict_proba(X_val)[:,1]
    val_rows = [calc_metrics(y_val, pv, t) for t in np.linspace(.05,.95,91)]
    best_val = max(val_rows, key=lambda m:(m["f1"],m["recall"],-m["fpr"]))
    lr_candidates.append((C,lr,best_val,pv))

selected_C, lr_model, lr_val_metrics, _ = max(
    lr_candidates, key=lambda row:(row[2]["f1"],row[2]["recall"],-row[2]["fpr"])
)
lr_test_prob = lr_model.predict_proba(X_test)[:,1]
lr_test_metrics = calc_metrics(y_test, lr_test_prob, lr_val_metrics["threshold"])
lr_test_metrics.update({"model":f"LogisticRegression_C{selected_C}", "validation_selection":lr_val_metrics})
print("Selected using Validation only:", selected_C, lr_val_metrics)
print("Test result:", lr_test_metrics)


## 5. Bảng so sánh URL trên Test

Chỉ các kết quả thực sự chạy thành công mới được đưa vào bảng.


In [ ]:
url_comparison = pd.DataFrame([
    {"model": url_test_metrics["model"], **{k:v for k,v in url_test_metrics.items() if k in ["precision","recall","f1","fpr","pr_auc_average_precision","threshold_locked_from_validation"]}},
    {"model": lr_test_metrics["model"], **{k:v for k,v in lr_test_metrics.items() if k in ["precision","recall","f1","fpr","pr_auc_average_precision","threshold"]}},
])
display(url_comparison)
url_comparison.to_csv(REPORTS / "week9_url_test_comparison.csv", index=False)
(REPORTS / "week9_url_test_metrics.json").write_text(json.dumps({
    "xgboost":url_test_metrics, "logistic_regression":lr_test_metrics,
    "protocol":"Threshold and model selected on Validation; Test used only for final evaluation."
}, indent=2), encoding="utf-8")


## 6. Kiểm tra nhánh văn bản XLM-RoBERTa và baseline TF-IDF

Trong gói ZIP được rà soát, artifact văn bản hiện có là `text_model.joblib` với metadata `TFIDF_LR_source_binary`. Chưa thấy checkpoint XLM-RoBERTa, tokenizer, cấu hình 3 lớp hoặc artifact inference tương ứng.

Cell này chỉ kiểm tra sự tồn tại. Không suy diễn kết quả XLM-R từ baseline.


In [ ]:
text_artifact = MODELS / "text_model.joblib"
xlmr_candidates = [MODELS / "xlmr", MODELS / "xlm_roberta", MODELS / "xlmr_text", MODELS / "xlmr_binary", PROJECT_ROOT / "checkpoints" / "xlm-roberta"]
xlmr_found = [str(p) for p in xlmr_candidates if p.exists()]
text_model = None
text_model_fitted = False
text_model_error = None
if text_artifact.exists():
    try:
        text_model = joblib.load(text_artifact)
        from sklearn.utils.validation import check_is_fitted
        check_is_fitted(text_model)
        # Pipeline-level check may miss a malformed inner transformer; smoke-test inference too.
        text_model.predict_proba([text_input("kiem tra pipeline")])
        text_model_fitted = True
    except Exception as exc:
        text_model_error = f"{type(exc).__name__}: {exc}"
text_status = {
    "text_artifact_exists": text_artifact.exists(),
    "text_model_fitted": text_model_fitted,
    "text_model_error": text_model_error,
    "text_model_manifest_name": manifest.get("text_model"),
    "xlmr_checkpoint_candidates_found": xlmr_found,
    "current_text_task": "binary source Spam/Scam proxy",
    "status": "READY" if text_model_fitted else "BLOCKED: run scripts/repair_text_model.py --repair after checking the project root",
}
print(json.dumps(text_status, ensure_ascii=False, indent=2))


## 6A. Đánh giá Text-only trên SMS Validation và Test

Ghép `full_dataset.csv` với `sms_split_manifest.csv` theo `message_id`. Chỉ sử dụng threshold trong manifest (được chọn trên Validation). Không fit hoặc chọn threshold trên Test. Nhãn 1 là Spam/Scam gộp chung, không phải nhãn phishing đã xác minh.


In [ ]:
if text_model_fitted:
    raw_sms_path = DATA / "raw" / "full_dataset.csv"
    sms_split_path = DATA / "sms_split_manifest.csv"
    if not raw_sms_path.exists() or not sms_split_path.exists():
        print("BLOCKED: cần data/raw/full_dataset.csv và data/sms_split_manifest.csv")
    else:
        raw_sms = pd.read_csv(raw_sms_path)
        sms_split = pd.read_csv(sms_split_path)
        sms_df = raw_sms.merge(sms_split, on="message_id", how="inner", validate="one_to_one")
        assert len(sms_df) == len(sms_split), "Không ghép đủ message_id"
        group_sets = [set(sms_df.loc[sms_df.split.eq(s), "group"]) for s in ["train", "validation", "test"]]
        assert not any(group_sets[i] & group_sets[j] for i in range(3) for j in range(i+1,3)), "Group leakage"
        positive_index = list(text_model.classes_).index(1)
        sms_threshold = float(manifest["text_threshold"])
        sms_rows = []
        for split_name in ["validation", "test"]:
            part = sms_df[sms_df.split.eq(split_name)].copy()
            X = part.message.map(text_input).tolist()
            t0 = time.perf_counter()
            probs = text_model.predict_proba(X)[:, positive_index]
            elapsed = time.perf_counter() - t0
            y = part.label.astype(int).to_numpy()
            pred = (probs >= sms_threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
            sms_rows.append({"model":"TFIDF_LR", "split":split_name, "n":len(y), "threshold_from_validation":sms_threshold,
                "precision":float(precision_score(y,pred,zero_division=0)), "recall":float(recall_score(y,pred,zero_division=0)),
                "f1":float(f1_score(y,pred,zero_division=0)), "fpr":float(fp/max(1,fp+tn)),
                "pr_auc_average_precision":float(average_precision_score(y,probs)),
                "confusion_matrix_TN_FP_FN_TP":[[int(tn),int(fp)],[int(fn),int(tp)]],
                "total_inference_seconds":float(elapsed), "mean_inference_ms_per_sms":float(elapsed*1000/max(1,len(y)))})
        sms_text_metrics = pd.DataFrame(sms_rows)
        display(sms_text_metrics)
        sms_text_metrics.to_csv(REPORTS / "week9_sms_text_metrics.csv", index=False)
        (REPORTS / "week9_sms_text_metrics.json").write_text(json.dumps(sms_rows,ensure_ascii=False,indent=2),encoding="utf-8")
else:
    print("BLOCKED: text_model.joblib chưa fitted; chạy scripts/repair_text_model.py --repair trước khi đánh giá.")


### Điều kiện để hoàn thành so sánh văn bản

Cần bổ sung đầy đủ: checkpoint `xlm-roberta-base` đã fine-tune, tokenizer/config, nhãn ba lớp đã rà soát và tập Test có ground truth tương ứng. Nếu chỉ có nhãn gốc Legitimate/Spam-Scam nhị phân thì không thể tự tách Spam/Advertisement khỏi Phishing/Scam bằng suy đoán.

Khi đủ dữ liệu, phải huấn luyện baseline TF-IDF + Logistic Regression và XLM-RoBERTa trên cùng tập Train, chọn hyperparameter/threshold trên Validation, rồi đánh giá cả hai trên cùng Test.


## 7. Risk Fusion — kiểm tra tính hợp lệ của benchmark

Manifest hiện ghi alpha = 1.0, target = `source_spam_or_scam`, status = `proxy_only_not_validated_phishing_probability`. Điều này không phải bằng chứng Text + URL cải thiện phát hiện phishing.

Để chạy tuning Risk Fusion đúng đề cương, cần file paired Validation có các cột `sample_id, split, y_phishing, p_text, p_url, has_url, label_status`, trong đó mọi nhãn `label_status` phải là `human_reviewed`. Không dùng Test để chọn alpha hoặc threshold.


In [ ]:
fusion_meta = manifest.get("fusion", {})
fusion_audit = {
    "alpha_current": fusion_meta.get("alpha"),
    "threshold_current": fusion_meta.get("threshold"),
    "target_current": fusion_meta.get("target"),
    "status_current": fusion_meta.get("status"),
    "paired_validation_rows": fusion_meta.get("paired_validation_rows"),
    "valid_phishing_fusion_benchmark": (
        fusion_meta.get("status") == "human_reviewed_phishing_validation"
        and fusion_meta.get("target") == "phishing"
    )
}
print(json.dumps(fusion_audit, ensure_ascii=False, indent=2))
if not fusion_audit["valid_phishing_fusion_benchmark"]:
    print("BLOCKED: không báo cáo chỉ số Fusion hiện tại là kết quả phishing đã được xác nhận.")


### 7A. Text + URL trên SMS ghép cặp — chỉ là benchmark proxy

SMS có cả văn bản và URL nên có thể kiểm tra kỹ thuật ghép cặp. Tuy nhiên, nhãn SMS là `source_spam_or_scam`, còn URL model được huấn luyện cho `phishing`; đây không phải ground truth phishing ghép cặp. Kết quả chỉ dùng để khảo sát sự không tương thích mục tiêu, không được tuyên bố là mô hình Fusion phishing đã được xác thực. Alpha và threshold chỉ chọn trên Validation; Test chỉ đánh giá sau đó.


In [ ]:
if text_model_fitted and "sms_df" in globals() and "url_model" in globals():
    paired_cache = {}
    for split_name in ["validation", "test"]:
        part = sms_df[sms_df.split.eq(split_name)].copy()
        p_text = text_model.predict_proba(part.message.map(text_input).tolist())[:, list(text_model.classes_).index(1)]
        p_url, has_url = [], []
        for message in part.message.astype(str):
            found = extract_urls(message)
            has_url.append(bool(found))
            p_url.append(float(url_model.predict_proba(feature_frame(found))[:,1].max()) if found else 0.0)
        paired_cache[split_name] = (part.label.astype(int).to_numpy(), p_text, np.asarray(p_url), np.asarray(has_url))
    yv, ptv, puv, huv = paired_cache["validation"]
    fusion_candidates = []
    for alpha in np.linspace(0,1,21):
        p = np.where(huv, alpha*ptv+(1-alpha)*puv, ptv)
        for threshold in np.linspace(.05,.95,91):
            pred = p >= threshold
            tn,fp,fn,tp = confusion_matrix(yv,pred,labels=[0,1]).ravel()
            f1v = f1_score(yv,pred,zero_division=0); recallv=recall_score(yv,pred,zero_division=0); fprv=fp/max(1,fp+tn)
            fusion_candidates.append((f1v,recallv,-fprv,float(alpha),float(threshold)))
    _,_,_,selected_alpha,selected_threshold = max(fusion_candidates)
    yt, ptt, put, hut = paired_cache["test"]
    p_test_fusion = np.where(hut, selected_alpha*ptt+(1-selected_alpha)*put, ptt)
    pred = p_test_fusion >= selected_threshold
    tn,fp,fn,tp = confusion_matrix(yt,pred,labels=[0,1]).ravel()
    proxy_result = {"target":"source_spam_or_scam_proxy_not_phishing_ground_truth", "validation_selected_alpha":selected_alpha, "validation_selected_threshold":selected_threshold,
        "test_rows":int(len(yt)), "test_rows_with_url":int(hut.sum()), "precision":float(precision_score(yt,pred,zero_division=0)),
        "recall":float(recall_score(yt,pred,zero_division=0)), "f1":float(f1_score(yt,pred,zero_division=0)),
        "fpr":float(fp/max(1,fp+tn)), "pr_auc_average_precision":float(average_precision_score(yt,p_test_fusion)),
        "confusion_matrix_TN_FP_FN_TP":[[int(tn),int(fp)],[int(fn),int(tp)]],
        "status":"exploratory_only_label_semantics_mismatch"}
    print(json.dumps(proxy_result,ensure_ascii=False,indent=2))
    (REPORTS/"week9_sms_text_url_proxy.json").write_text(json.dumps(proxy_result,ensure_ascii=False,indent=2),encoding="utf-8")
else:
    print("BLOCKED: cần text model fitted và SMS Test/Validation đã ghép.")


## 8. Robustness — phép thử độ nhạy, không tự gán ground truth

Các phép biến đổi dưới đây tạo bản thử nghiệm có dấu/không dấu, viết tắt, lỗi chính tả và ký tự chèn. Khi chưa có nhãn ground truth cho từng biến thể, chỉ báo cáo mức thay đổi xác suất (probability shift), không tính Accuracy/F1 hoặc Performance Drop.


In [ ]:
def remove_vietnamese_diacritics(text):
    text = unicodedata.normalize("NFD", str(text))
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ","d").replace("Đ","D")

def robustness_variants(text):
    s = str(text)
    no_diacritics = remove_vietnamese_diacritics(s)
    abbreviations = (s.replace("quý khách","q.k").replace("vui lòng","vui long")
                       .replace("ngân hàng","NH").replace("tài khoản","tk"))
    typo = s.replace("đăng nhập","đăg nhập").replace("xác minh","xác mih")
    inserted = re.sub(r"([a-zA-Z])", r"\1.", s, count=4)
    return {
        "original":s, "no_diacritics":no_diacritics,
        "abbreviation":abbreviations, "typo":typo, "inserted_chars":inserted
    }

# Chỉ thực hiện nếu text artifact có predict_proba; nhãn/đầu ra hiện tại là Spam/Scam proxy.
if text_model_fitted:
    # Reuse the checked artifact loaded in Cell 14; do not silently load a different model.
    text_model = text_model
    examples = [
        "Vui lòng xác minh tài khoản ngân hàng của bạn ngay hôm nay.",
        "Chúc mừng bạn nhận được phần thưởng. Truy cập liên kết để xác nhận.",
        "Chiều nay chúng ta gặp nhau lúc 15 giờ nhé."
    ]
    records = []
    for i, message in enumerate(examples, start=1):
        for variant_name, variant_text in robustness_variants(message).items():
            t0 = time.perf_counter()
            # Same normalization and positive-class lookup as production inference.
            probability = float(text_model.predict_proba([text_input(variant_text)])[0, list(text_model.classes_).index(1)])
            latency_ms = (time.perf_counter()-t0)*1000
            records.append({"case":i,"variant":variant_name,"p_source_spam_scam":probability,"latency_ms":latency_ms})
    robustness_df = pd.DataFrame(records)
    display(robustness_df)
    robustness_df.to_csv(REPORTS / "week9_text_robustness_sensitivity.csv", index=False)
else:
    print("BLOCKED: text_model.joblib chưa fitted; chạy scripts/repair_text_model.py --repair trước khi kiểm thử robustness.")


## 9. Robustness URL và thời gian suy luận

Đây là phép thử độ nhạy của xác suất với một số biến thể URL. Các biến thể không có nhãn ground truth độc lập nên không được dùng để tuyên bố độ chính xác.


In [ ]:
url_examples = [
    "https://example.com/login",
    "http://secure-account.example.org/verify?user=123",
]
url_variants = []
for original in url_examples:
    variants = {
        "original": original,
        "https_scheme": original.replace("http://","https://",1),
        "extra_path": original.rstrip("/") + "/account/verify",
        "query_added": original + ("&" if "?" in original else "?") + "ref=campaign123"
    }
    for name, value in variants.items():
        try:
            frame = feature_frame([value])
            t0 = time.perf_counter()
            prob = float(url_model.predict_proba(frame)[0,1])
            latency = (time.perf_counter()-t0)*1000
            url_variants.append({"original":original,"variant":name,"url":value,
                                 "p_phishing":prob,"inference_ms":latency})
        except Exception as exc:
            url_variants.append({"original":original,"variant":name,"url":value,"error":str(exc)})
url_robustness_df = pd.DataFrame(url_variants)
display(url_robustness_df)
url_robustness_df.to_csv(REPORTS / "week9_url_robustness_sensitivity.csv", index=False)


## 10. Kiểm thử API, đầu vào và phiên đăng nhập

Các bài kiểm thử API hiện có nằm trong `tests/test_system.py`; báo cáo tuần 5–8 ghi nhận 9 test pass. Chạy lại trong môi trường Python 3.12 của dự án:

```powershell
python -m pytest -q
```

Các tình huống cần xác nhận trong tuần 9:
- Nội dung rỗng, sai kiểu, vượt giới hạn độ dài.
- Nhiều URL (giới hạn hiện tại của engine là tối đa 20 URL).
- URL sai định dạng.
- Cookie/phiên đăng nhập hết hạn hoặc bị thu hồi.
- Quyền truy cập sau khi user bị khóa hoặc đổi vai trò.
- Thời gian suy luận API end-to-end và thời gian inference model riêng.

Không thể xác nhận phiên đăng nhập hết hạn chỉ từ notebook offline; cần chạy server và thực hiện test API/session có kiểm soát. Ghi lại mã HTTP, response body và thời gian.


In [ ]:
# Chạy lại bộ test hiện hữu từ terminal tại PROJECT_ROOT:
print("Project root:", PROJECT_ROOT)
print("Command: python -m pytest -q")
print("Không tự động gọi API đang chạy trong cell này để tránh ghi dữ liệu vào database của bạn.")


## 11. Tổng hợp trạng thái và xuất báo cáo

Kết quả URL Test sẽ được lưu trong `reports/`. Các mục XLM-R, Text Test và Fusion chỉ được đánh dấu hoàn thành sau khi có checkpoint/ground truth hợp lệ.


In [ ]:
summary = {
    "week": 9,
    "url_test_evaluated": "url_test_metrics" in globals(),
    "sms_text_test_evaluated": "sms_text_metrics" in globals(),
    "xlmr_status": "prepared_binary_training_script; execution requires transformers/torch and model download",
    "text_task": "binary source Legitimate vs Spam/Scam; not a 3-class phishing classifier",
    "fusion_status": "exploratory_proxy_only; label semantics mismatch; not validated phishing fusion",
    "robustness_status": "sensitivity-only where transformed samples lack independent ground truth",
    "test_tuning_policy": "No model/threshold/alpha tuning on Test"
}
(REPORTS / "week9_execution_status.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))
